# Intercoder Agreement — MCS-Aligned Multiclass Nodepair Kappa

Uses the same MCS greedy mapping from the parallel graph evaluation to align nodes
between raters, then computes Cohen's kappa (pairwise) and Fleiss' kappa (3-rater) on a
**5-category multiclass** label for every undirected node pair among mapped nodes:
`none | hierarchy | directional | correlational | moderation`.

Preprocessing is identical to `graph_structural_eval_parallel.ipynb`.

In [ ]:
import json
import random
import re
from pathlib import Path

import networkx as nx
import pandas as pd

print('Imports OK')

In [ ]:
# ---- Config ----
ROOT = Path('..').resolve() if Path('..').joinpath('data').exists() else Path('.').resolve()

A_CSV   = ROOT / 'data' / 'raw' / 'annotation_csv' / 'coder_A_r.csv'
B_CSV   = ROOT / 'data' / 'raw' / 'annotation_csv' / 'coder_B_r.csv'
GT_JSON = ROOT / 'data' / 'raw' / 'EmpiriGraph-Psy_gold_annotation.json'
OUTPUT_DIR = ROOT / 'data' / 'output' / 'intercoder'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORDERS = ['degree_desc', 'degree_asc', 'label', 'random_11', 'random_29']
CATS   = ['none', 'hierarchy', 'directional', 'correlational', 'moderation']

print('ROOT:      ', ROOT)
print('A_CSV:     ', A_CSV)
print('B_CSV:     ', B_CSV)
print('GT_JSON:   ', GT_JSON)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# ---- Parsers ----

def _safe(v):
    return '' if pd.isna(v) else str(v).strip()


def _smart_split(text: str, delim: str = ';') -> list:
    raw = text.split(delim)
    result, buf, depth = [], '', 0
    for part in raw:
        buf = (buf + delim + part) if buf else part
        depth += part.count('(') - part.count(')')
        if depth <= 0:
            result.append(buf.strip())
            buf = ''
            depth = 0
    if buf:
        result.append(buf.strip())
    return [x for x in result if x]


def _validation_from_text(text: str, default: str = 'hypothesized') -> str:
    t = str(text).lower()
    if 'validated' in t: return 'validated'
    if 'null' in t: return 'null'
    if 'hypothesized' in t or 'hypothesis' in t: return 'hypothesized'
    return default


def _edge_type(label: str) -> str:
    l = str(label).lower()
    if l == 'hierarchy': return 'hierarchy'
    if l.startswith('direction'): return 'directional'
    if l.startswith('correlation'): return 'correlational'
    if l.startswith('moderation'): return 'moderation'
    return 'unknown'


def parse_gt_graph(article: dict) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    ann = article.get('annotations', [{}])[0]
    vm = {}
    for r in ann.get('result', []):
        if r.get('type') == 'labels' and 'Variable' in r.get('value', {}).get('labels', []):
            vm[r['id']] = r.get('value', {}).get('text', '').strip()
    for r in ann.get('result', []):
        if r.get('type') == 'textarea' and r.get('from_name') == 'var_name' and r.get('id') in vm:
            txt = r.get('value', {}).get('text', '')
            if isinstance(txt, list): txt = txt[0] if txt else ''
            if str(txt).strip(): vm[r['id']] = str(txt).strip()
    for rid, name in vm.items():
        G.add_node(rid, label=name)
    for r in ann.get('result', []):
        if r.get('type') == 'relation':
            fid, tid = r.get('from_id'), r.get('to_id')
            if fid in vm and tid in vm:
                for lbl in r.get('labels', []):
                    et = _edge_type(lbl)
                    if et != 'unknown':
                        G.add_edge(fid, tid, edge_type=et,
                                   validation=_validation_from_text(lbl, default='validated'))
    return G


def parse_coder_row(row) -> nx.MultiDiGraph:
    """Parse coder CSV row into a graph.
    Moderation format: 'moderator moderates (var1, var2, ...) [val]'
    Each var_i produces a moderator -> var_i edge.
    Correlational edges use <-> and are treated as undirected after preprocessing."""
    G = nx.MultiDiGraph()
    seen = set()

    def _ensure(n):
        if n and n not in G: G.add_node(n, label=n)

    def _add_edge(s, t, et, validation='hypothesized'):
        k = (s, t, et, validation)
        if k not in seen:
            G.add_edge(s, t, edge_type=et, validation=validation)
            seen.add(k)

    for v in _smart_split(_safe(row.get('variables', ''))): _ensure(v)

    # Hierarchy
    for e in _smart_split(_safe(row.get('hierarchy', ''))):
        e2 = _strip_bracket_suffix(e)
        if '->' in e2:
            p = e2.split('->', 1)
            s, t = p[0].strip(), p[1].strip()
            if s and t: _ensure(s); _ensure(t); _add_edge(s, t, 'hierarchy', 'validated')

    # Directional
    for e in _smart_split(_safe(row.get('directional', ''))):
        m = re.search(r'\[(validated|null|hypothesized)\]$', e.strip())
        val = m.group(1) if m else 'hypothesized'
        if m: e = e[:m.start()].strip()
        e = _strip_bracket_suffix(e)
        if '->' in e:
            p = e.split('->', 1)
            s, t = p[0].strip(), p[1].strip()
            if s and t: _ensure(s); _ensure(t); _add_edge(s, t, 'directional', val)

    # Correlational (undirected after preprocessing via canonicalise_corr)
    for e in _smart_split(_safe(row.get('correlational', ''))):
        m = re.search(r'\[(validated|null|hypothesized)\]$', e.strip())
        val = m.group(1) if m else 'hypothesized'
        if m: e = e[:m.start()].strip()
        e = _strip_bracket_suffix(e)
        if '<->' in e:
            p = e.split('<->', 1)
            s, t = p[0].strip(), p[1].strip()
            if s and t: _ensure(s); _ensure(t); _add_edge(s, t, 'correlational', val)

    # Moderation: 'moderator moderates (var1, var2, ...) [val]'
    # Each comma-separated variable produces: moderator -> var_i
    for e in _smart_split(_safe(row.get('moderation', ''))):
        m_val = re.search(r'\[(validated|null|hypothesized)\]$', e.strip())
        val = m_val.group(1) if m_val else 'hypothesized'
        if m_val: e = e[:m_val.start()].strip()
        m_mod = re.match(r'^(.+?)\s+moderates?\s*\((.+)\)\s*$', e.strip(),
                         re.IGNORECASE | re.DOTALL)
        if m_mod:
            moderator = m_mod.group(1).strip().rstrip(',').strip()
            vars_str  = m_mod.group(2)
            # split on commas respecting nested parentheses
            mod_vars = []
            buf, depth = '', 0
            for ch in vars_str:
                if ch == '(':   depth += 1; buf += ch
                elif ch == ')': depth -= 1; buf += ch
                elif ch == ',' and depth == 0:
                    v = buf.strip()
                    if v: mod_vars.append(v)
                    buf = ''
                else: buf += ch
            if buf.strip(): mod_vars.append(buf.strip())
            _ensure(moderator)
            for mv in mod_vars:
                mv = mv.strip()
                if mv: _ensure(mv); _add_edge(moderator, mv, 'moderation', val)

    return G


print('Parsers defined')

In [ ]:
# ---- Preprocessing (identical to graph_eval.ipynb) ----

def canonicalise_corr(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    H = nx.MultiDiGraph(); seen = set()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    def _lbl(n): return G.nodes[n].get('label', str(n))
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et == 'correlational':
            lu, lv = _lbl(u), _lbl(v)
            cu, cv = (v, u) if (lu, str(u)) > (lv, str(v)) else (u, v)
            k = (cu, cv, et)
            if k not in seen: H.add_edge(cu, cv, **dict(d)); seen.add(k)
        else:
            k = (u, v, et)
            if k not in seen: H.add_edge(u, v, **dict(d)); seen.add(k)
    return H


def merge_single_lv_hierarchy(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    children = {}
    for u, v, d in G.edges(data=True):
        if d.get('edge_type') == 'hierarchy': children.setdefault(u, []).append(v)
    rename, drop_hier = {}, set()
    for hv, lvs in children.items():
        if len(lvs) == 1:
            lv = lvs[0]
            merged = f"{str(G.nodes[hv].get('label', hv)).strip()} ({str(G.nodes[lv].get('label', lv)).strip()})"
            rename[hv] = merged; rename[lv] = merged; drop_hier.add((hv, lv))
    def mapped(n): return rename.get(n, n)
    H = nx.MultiDiGraph(); seen = set()
    for n, d in G.nodes(data=True):
        mn = mapped(n)
        if mn not in H: H.add_node(mn, label=mn if n in rename else d.get('label', mn))
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et == 'hierarchy' and (u, v) in drop_hier: continue
        mu, mv = mapped(u), mapped(v)
        if mu == mv: continue
        k = (mu, mv, et)
        if k not in seen: H.add_edge(mu, mv, **dict(d)); seen.add(k)
    return H


def propagate_to_upper(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    H = G.copy()
    while True:
        added = 0
        hier_pairs = [(u, v) for u, v, d in H.edges(data=True) if d.get('edge_type') == 'hierarchy']
        existing = {(u, v, d.get('edge_type')) for u, v, d in H.edges(data=True)}
        for upper, lower in hier_pairs:
            for _, tgt, d in list(H.out_edges(lower, data=True)):
                et = d.get('edge_type')
                if et != 'hierarchy' and (upper, tgt, et) not in existing:
                    H.add_edge(upper, tgt, **d); existing.add((upper, tgt, et)); added += 1
            for src, _, d in list(H.in_edges(lower, data=True)):
                et = d.get('edge_type')
                if et != 'hierarchy' and (src, upper, et) not in existing:
                    H.add_edge(src, upper, **d); existing.add((src, upper, et)); added += 1
        if added == 0: break
    return H


def drop_nonhierarchy_between_hierarchy_linked(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    H = nx.MultiDiGraph(); seen = set()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    hier_pairs = {frozenset((u, v)) for u, v, d in G.edges(data=True)
                  if d.get('edge_type') == 'hierarchy' and u != v}
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et != 'hierarchy' and frozenset((u, v)) in hier_pairs: continue
        k = (u, v, et)
        if k not in seen: H.add_edge(u, v, **dict(d)); seen.add(k)
    return H


def drop_corr_when_direction_exists(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    H = nx.MultiDiGraph(); seen = set()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    dir_pairs = {(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'directional'}
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et == 'correlational' and ((u, v) in dir_pairs or (v, u) in dir_pairs): continue
        k = (u, v, et)
        if k not in seen: H.add_edge(u, v, **dict(d)); seen.add(k)
    return H


def collapse_multirel_edges_by_priority(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    priority = {
        ('directional', 'validated'): 0, ('directional', 'null'): 1, ('directional', 'hypothesized'): 2,
        ('correlational', 'validated'): 3, ('correlational', 'null'): 4, ('correlational', 'hypothesized'): 5,
        ('moderation', 'validated'): 6, ('moderation', 'null'): 7, ('moderation', 'hypothesized'): 8,
    }
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    seen_h = set()
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et == 'hierarchy':
            k = (u, v, et)
            if k not in seen_h: H.add_edge(u, v, **dict(d)); seen_h.add(k)
    best = {}
    for u, v, d in G.edges(data=True):
        et = d.get('edge_type', '')
        if et == 'hierarchy': continue
        val = _validation_from_text(d.get('validation', 'hypothesized'))
        rank = priority.get((et, val), 10_000)
        cand = (rank, et, val, dict(d))
        pair = (u, v)
        if pair not in best or cand < best[pair]: best[pair] = cand
    for (u, v), (_, et, val, attrs) in best.items():
        attrs = dict(attrs); attrs['edge_type'] = et; attrs['validation'] = val
        H.add_edge(u, v, **attrs)
    return H


def preprocess_graph_for_eval(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    G = canonicalise_corr(G)
    G = merge_single_lv_hierarchy(G)
    G = propagate_to_upper(G)
    G = drop_nonhierarchy_between_hierarchy_linked(G)
    G = canonicalise_corr(G)
    G = drop_corr_when_direction_exists(G)
    G = collapse_multirel_edges_by_priority(G)
    return G

print('Preprocessing defined')

In [ ]:
# ---- MCS Greedy Mapping (same logic as graph eval parallel) ----

def _ordered_nodes(G, strategy):
    nodes = list(G.nodes())
    if strategy == 'degree_desc':
        return sorted(nodes, key=lambda n: G.degree(n), reverse=True)
    if strategy == 'degree_asc':
        return sorted(nodes, key=lambda n: G.degree(n))
    if strategy == 'label':
        return sorted(nodes, key=lambda n: str(G.nodes[n].get('label', n)))
    if strategy.startswith('random_'):
        seed = int(strategy.split('_', 1)[1])
        rnd = random.Random(seed)
        out = nodes[:]
        rnd.shuffle(out)
        return out
    raise ValueError(strategy)


def best_greedy_mapping(G1: nx.MultiDiGraph, G2: nx.MultiDiGraph) -> dict:
    """Try all ORDERS greedy mappings G1->G2, return best by edge matches."""
    pred_nodes = list(G2.nodes())
    pred_edge = {(u, v, d['edge_type']) for u, v, d in G2.edges(data=True)}
    adj_out = {n: [(v, d['edge_type']) for _, v, d in G1.out_edges(n, data=True)] for n in G1.nodes()}
    adj_in = {n: [(u, d['edge_type']) for u, _, d in G1.in_edges(n, data=True)] for n in G1.nodes()}

    def cnt(gn, pn, phi):
        c = 0
        for v, t in adj_out.get(gn, []):
            pv = phi.get(v)
            if pv and (pn, pv, t) in pred_edge: c += 1
        for u, t in adj_in.get(gn, []):
            pu = phi.get(u)
            if pu and (pu, pn, t) in pred_edge: c += 1
        return c

    best_em, best_phi = -1, {}
    for strat in ORDERS:
        gt_nodes = _ordered_nodes(G1, strat)
        phi, avail, em = {}, pred_nodes[:], 0
        for gn in gt_nodes:
            bc, bp = -1, None
            for p in avail:
                c = cnt(gn, p, phi)
                if c > bc: bc = c; bp = p
            if bp is not None:
                phi[gn] = bp; avail.remove(bp); em += bc
        if em > best_em:
            best_em = em; best_phi = dict(phi)
    return best_phi

print('Mapping defined')

In [ ]:
# ---- Kappa Functions ----

def rel_for_pair(G, u, v):
    """Multiclass label for undirected node pair.
    Priority: directional > correlational > moderation > hierarchy > none."""
    pr = {'directional': 0, 'correlational': 1, 'moderation': 2, 'hierarchy': 3}
    best = ('none', 999)
    for a, b in [(u, v), (v, u)]:
        data = G.get_edge_data(a, b, default={})
        for _, d in data.items():
            e = d.get('edge_type', '')
            r = pr.get(e, 999)
            if r < best[1]: best = (e, r)
    return best[0]


def cohen_multiclass(l1, l2):
    n = len(l1)
    if n == 0: return float('nan')
    cats = sorted(set(l1) | set(l2))
    idx = {c: i for i, c in enumerate(cats)}; nc = len(cats)
    conf = [[0] * nc for _ in range(nc)]
    for a, b in zip(l1, l2): conf[idx[a]][idx[b]] += 1
    po = sum(conf[i][i] for i in range(nc)) / n
    p1 = [sum(conf[i][j] for j in range(nc)) / n for i in range(nc)]
    p2 = [sum(conf[i][j] for i in range(nc)) / n for j in range(nc)]
    pe = sum(p1[i] * p2[i] for i in range(nc))
    return 1.0 if abs(1 - pe) < 1e-12 else (po - pe) / (1 - pe)


def fleiss_multiclass(rater_labels):
    m = len(rater_labels); N = len(rater_labels[0]) if m else 0
    if m < 2 or N == 0: return float('nan')
    cats = sorted(set(x for v in rater_labels for x in v))
    cidx = {c: j for j, c in enumerate(cats)}; k = len(cats)
    mat = []
    for i in range(N):
        row = [0] * k
        for v in rater_labels: row[cidx[v[i]]] += 1
        mat.append(row)
    P = [(sum(v * v for v in row) - m) / (m * (m - 1)) for row in mat]
    Pbar = sum(P) / N
    pj = [sum(row[j] for row in mat) / (N * m) for j in range(k)]
    Pe = sum(p * p for p in pj)
    return 1.0 if abs(1 - Pe) < 1e-12 else (Pbar - Pe) / (1 - Pe)

print('Kappa functions defined')

In [ ]:
# ---- Load and Preprocess ----

A_df = pd.read_csv(A_CSV)
B_df = pd.read_csv(B_CSV)
with open(GT_JSON, 'r', encoding='utf-8') as f:
    gt_data = json.load(f)

A_raw = {int(r['task_id']): parse_coder_row(r) for _, r in A_df.iterrows() if not pd.isna(r.get('task_id'))}
B_raw = {int(r['task_id']): parse_coder_row(r) for _, r in B_df.iterrows() if not pd.isna(r.get('task_id'))}
GT_raw = {int(a['id']): parse_gt_graph(a) for a in gt_data if a.get('id') is not None}

ids = sorted(set(A_raw) & set(B_raw) & set(GT_raw))
print(f'Overlap articles (A ∩ B ∩ GT): {len(ids)}')

A = {i: preprocess_graph_for_eval(A_raw[i]) for i in ids}
B = {i: preprocess_graph_for_eval(B_raw[i]) for i in ids}
J = {i: preprocess_graph_for_eval(GT_raw[i]) for i in ids}

print('Preprocessed all graphs')

In [ ]:
# ---- Compute Cohen (pairwise) ----
# For each pair (X, Y): map X->Y via MCS greedy.
# For ALL undirected pairs among mapped X-nodes,
# assign 5-category label in X and in Y (via mapped nodes).

cohen_rows = []
for name, X, Y in [('A-B', A, B), ('A-JSON', A, J), ('B-JSON', B, J)]:
    lx, ly, sum_pairs = [], [], 0
    for aid in ids:
        g1, g2 = X[aid], Y[aid]
        phi = best_greedy_mapping(g1, g2)
        mapped_nodes = list(phi.keys())
        for i, u in enumerate(mapped_nodes):
            for j in range(i + 1, len(mapped_nodes)):
                v = mapped_nodes[j]
                lx.append(rel_for_pair(g1, u, v))
                ly.append(rel_for_pair(g2, phi[u], phi[v]))
                sum_pairs += 1
    cohen_rows.append({
        'pair': name,
        'kappa': cohen_multiclass(lx, ly),
        'n_node_pairs': sum_pairs,
        'n_categories': 5,
        'categories': '|'.join(CATS),
    })

cohen_df = pd.DataFrame(cohen_rows)
cohen_csv = OUTPUT_DIR / 'cohen_kappa_mcs_multiclass_allpairs.csv'
cohen_df.to_csv(cohen_csv, index=False)
print('Saved:', cohen_csv)
cohen_df

In [ ]:
# ---- Compute Fleiss (3-rater) ----
# Pivot on GT: map GT->A and GT->B.
# For ALL undirected pairs among GT nodes mapped to both A and B,
# assign 5-category label in GT, A (via mapped), B (via mapped).

lj, la, lb, sum_pairs = [], [], [], 0
for aid in ids:
    gJ, gA, gB = J[aid], A[aid], B[aid]
    phiA = best_greedy_mapping(gJ, gA)  # GT -> A
    phiB = best_greedy_mapping(gJ, gB)  # GT -> B
    gt_nodes = [n for n in gJ.nodes() if n in phiA and n in phiB]
    for i, u in enumerate(gt_nodes):
        for j in range(i + 1, len(gt_nodes)):
            v = gt_nodes[j]
            lj.append(rel_for_pair(gJ, u, v))
            la.append(rel_for_pair(gA, phiA[u], phiA[v]))
            lb.append(rel_for_pair(gB, phiB[u], phiB[v]))
            sum_pairs += 1

fleiss_rows = [{
    'measure': 'fleiss_mcs_multiclass_allpairs',
    'kappa': fleiss_multiclass([lj, la, lb]),
    'n_node_pairs': sum_pairs,
    'n_categories': 5,
    'categories': '|'.join(CATS),
}]
fleiss_df = pd.DataFrame(fleiss_rows)
fleiss_csv = OUTPUT_DIR / 'fleiss_kappa_mcs_multiclass_allpairs.csv'
fleiss_df.to_csv(fleiss_csv, index=False)
print('Saved:', fleiss_csv)
fleiss_df

In [ ]:
# ---- Label Distribution Check ----
from collections import Counter

for name, X, Y in [('A-B', A, B), ('A-JSON', A, J), ('B-JSON', B, J)]:
    lx, ly = [], []
    for aid in ids:
        phi = best_greedy_mapping(X[aid], Y[aid])
        mn = list(phi.keys())
        for i, u in enumerate(mn):
            for j in range(i + 1, len(mn)):
                v = mn[j]
                lx.append(rel_for_pair(X[aid], u, v))
                ly.append(rel_for_pair(Y[aid], phi[u], phi[v]))
    print(f'{name}  X: {Counter(lx)}  Y: {Counter(ly)}')